# Event-Window Construction

This notebook is Phase 4 of the Delhi AQI / GRAP project: **Event-Window
Construction**.

**Scope of this notebook.** This notebook has exactly one job: build a single,
standardized table that lines up every verified GRAP event with the days
immediately before and after it, for every station. That table is the
`event_id x station_name x relative_day` grid described below, saved at the end
as `event_windows_master.csv`.

**Deliberately out of scope here.** No statistical analysis, no before-vs-after
comparison, no averages, no charts, no interpretation of pollution levels, and no
discussion of whether GRAP worked. This notebook only constructs and validates
the shape of the data. Every question about what the numbers mean is left for a
later notebook.

> The notebook assumes it is run from the `notebooks/` folder, so the data paths
> below start with `../`. This notebook does not modify any earlier notebook.

# Section 1 -- Load the Inputs

We need two files: the merged station-day dataset, and the manually verified
GRAP event calendar.

In [1]:
import pandas as pd

# Input 1: the merged Phase 2 dataset -- one row per station per day.
station_data_path = '../data/processed/station_daily_grap.csv'
station_df = pd.read_csv(station_data_path)
station_df['date'] = pd.to_datetime(station_df['date'])

# Input 2: the manually verified GRAP event calendar -- one row per official
# GRAP order that changed the active stage.
events_path = '../data/raw/grap/grap_events_manual.csv'
events_df = pd.read_csv(events_path)
events_df['effective_date'] = pd.to_datetime(events_df['effective_date'])

print('station_df rows:', len(station_df))
print('events_df rows:', len(events_df))
station_df.head()

station_df rows: 5840
events_df rows: 9


,station_id,station_name,date,year,pm25_ugm3,pm10_ugm3,air_temp_c,rh_pct,wind_speed_ms,wind_dir_deg,season,grap_stage,is_event_day,days_since_last_change,days_until_next_change,source_file
0,anand_vihar,Anand Vihar,2022-01-01,2022,393.56,598.82,13.97,75.32,0.32,230.10,2021-22,0,0,NaN,NaN,"raw_data_data_anand_vihar,_delhi_-_dpcc_1D_22.csv"
1,anand_vihar,Anand Vihar,2022-01-02,2022,424.81,598.95,14.33,75.32,0.32,230.36,2021-22,0,0,NaN,NaN,"raw_data_data_anand_vihar,_delhi_-_dpcc_1D_22.csv"
2,anand_vihar,Anand Vihar,2022-01-03,2022,375.52,636.12,14.79,72.44,0.35,223.51,2021-22,0,0,NaN,NaN,"raw_data_data_anand_vihar,_delhi_-_dpcc_1D_22.csv"
3,anand_vihar,Anand Vihar,2022-01-04,2022,291.86,564.58,16.14,66.95,0.31,209.49,2021-22,0,0,NaN,NaN,"raw_data_data_anand_vihar,_delhi_-_dpcc_1D_22.csv"
4,anand_vihar,Anand Vihar,2022-01-05,2022,353.05,449.37,14.45,86.60,0.89,138.08,2021-22,0,0,NaN,NaN,"raw_data_data_anand_vihar,_delhi_-_dpcc_1D_22.csv"


**What this does.** Loads the station-day dataset built in Phase 2 and converts
its `date` column to real dates. Separately loads the raw GRAP event calendar and
converts its `effective_date` column to real dates. Two prints confirm how many
rows each file contains before anything else happens.

**Why event alignment is useful.** Before any event window can be built, both
sides of the alignment -- the daily pollution/weather readings and the dated
list of GRAP events -- have to be loaded as real dates rather than plain text.
Real dates are what let us later ask "which calendar date is 7 days before this
event?" using simple date arithmetic.

**How analysts use event windows.** An event window is only meaningful once it
is anchored to an exact date. This first step exists purely so that anchor date
is available and trustworthy before we build anything on top of it.

In [2]:
# Keep only events that a human has verified against the official GRAP order.
# Per the project's data contract, only verified = 'Yes' rows are analysis-ready.
verified_events = events_df[events_df['verified'] == 'Yes'].copy()

# Keep just the two columns this notebook needs from the event calendar, and
# rename effective_date to event_date for clarity in the master table.
verified_events = verified_events[['event_id', 'effective_date']]
verified_events = verified_events.rename(columns={'effective_date': 'event_date'})
verified_events = verified_events.reset_index(drop=True)

print('Verified events available:', len(verified_events))
verified_events

Verified events available: 9


,event_id,event_date
0,E001,2022-10-05
1,E002,2022-10-19
2,E003,2022-10-29
3,E004,2022-11-03
4,E005,2022-11-06
5,E006,2022-11-14
6,E007,2022-12-04
7,E008,2022-12-07
8,E009,2022-12-30


**What this does.** Filters the event calendar down to rows marked
`verified = 'Yes'`, keeps only the event id and its effective date, and renames
that date column to `event_date`.

**Why event alignment is useful.** Every event window in this notebook will be
built around `event_date`. Filtering to verified events first means the windows
we build are only ever anchored to dates a human has already confirmed against
the official order -- not to a date that might still be provisional.

**How analysts use event windows.** Analysts only ever build windows around
confirmed events. Carrying an unverified event into the master table would mean
carrying an unconfirmed anchor date into every later analysis, so this filter is
applied once, at the very start.

# Section 2 -- Build the Event Window Grid

Before attaching any pollution or weather data, we first build an empty
"skeleton" table: one row for every combination of event, station, and relative
day. This guarantees every window has the same shape, regardless of what data is
or is not available for a particular day.

In [3]:
# The list of stations that should appear in every event window.
station_list = sorted(station_df['station_name'].unique())

# The 15 relative days that make up a +/-7 day window: -7, -6, ..., 0, ..., 6, 7.
relative_day_list = list(range(-7, 8))

print('Number of stations:', len(station_list))
print('Number of relative days per window:', len(relative_day_list))
print(station_list)
print(relative_day_list)

Number of stations: 8
Number of relative days per window: 15
['Anand Vihar', 'Bawana', 'Jawaharlal Nehru Stadium', 'Najafgarh', 'Narela', 'Okhla Phase-2', 'Punjabi Bagh', 'R K Puram']
[-7, -6, -5, -4, -3, -2, -1, 0, 1, 2, 3, 4, 5, 6, 7]


**What this does.** Builds two plain lists: every station name that appears in
the dataset, and the 15 relative-day numbers from -7 (seven days before the
event) to +7 (seven days after), with 0 representing the event day itself.

**Why event alignment is useful.** Relative days are what make different events
comparable to each other. An event's own calendar date is different every time,
but "3 days before the event" (`relative_day = -3`) means the same thing for
every event. Building this list up front fixes the shape every window must have.

**How analysts use event windows.** Analysts read event-window data by relative
day, not by calendar date, because relative day is what lines events up on a
shared axis. Fixing the list of relative days here is what makes that alignment
possible later.

In [4]:
# Build the skeleton one row at a time: for every verified event, for every
# relative day, for every station, create one row with the calendar date that
# relative day corresponds to.
skeleton_rows = []

for _, event_row in verified_events.iterrows():
    event_id = event_row['event_id']
    event_date = event_row['event_date']

    for relative_day in relative_day_list:
        calendar_date = event_date + pd.Timedelta(days=relative_day)

        for station_name in station_list:
            skeleton_rows.append({
                'event_id': event_id,
                'event_date': event_date,
                'relative_day': relative_day,
                'calendar_date': calendar_date,
                'station_name': station_name,
            })

# Turn the list of rows into a DataFrame -- this is the empty window skeleton.
window_skeleton = pd.DataFrame(skeleton_rows)

print('Skeleton rows built:', len(window_skeleton))
window_skeleton.head()

Skeleton rows built: 1080


,event_id,event_date,relative_day,calendar_date,station_name
0,E001,2022-10-05,-7,2022-09-28,Anand Vihar
1,E001,2022-10-05,-7,2022-09-28,Bawana
2,E001,2022-10-05,-7,2022-09-28,Jawaharlal Nehru Stadium
3,E001,2022-10-05,-7,2022-09-28,Najafgarh
4,E001,2022-10-05,-7,2022-09-28,Narela


**What this does.** Loops through every verified event, every relative day from
-7 to +7, and every station, and creates one row for each combination. Each row
records which event it belongs to, the event's own date, the relative day, the
actual calendar date that relative day falls on, and the station. The result is
a DataFrame with no pollution or weather values yet -- only the structure of the
windows.

**Why event alignment is useful.** Building the full grid first, before merging
in any real data, guarantees that every event window has exactly 15 days and
every day has exactly one row per station -- even if a particular day's
measurements later turn out to be missing. If we built the table by merging data
first, a day with no data would simply disappear instead of showing up as a
visible gap.

**How analysts use event windows.** This skeleton-first approach is standard in
event-study construction: fix the shape of the window before attaching data, so
that any missing data is visible as an explicit gap rather than a shortened or
uneven window.

# Section 3 -- Attach Pollution, Weather, and GRAP Data

With the skeleton in place, we now bring in the actual daily readings by
matching each skeleton row's station and calendar date to the station-day
dataset.

In [5]:
# Attach the real measurements by matching (station_name, calendar_date) in the
# skeleton to (station_name, date) in the station-day dataset. A left merge
# keeps every skeleton row, even if no matching data row exists.
event_windows = window_skeleton.merge(
    station_df[[
        'station_name', 'date', 'pm25_ugm3', 'pm10_ugm3',
        'air_temp_c', 'rh_pct', 'wind_speed_ms', 'wind_dir_deg', 'grap_stage'
    ]],
    left_on=['station_name', 'calendar_date'],
    right_on=['station_name', 'date'],
    how='left',
)

# The 'date' column duplicates 'calendar_date' after the merge, so it is dropped.
event_windows = event_windows.drop(columns=['date'])

print('event_windows rows after merge:', len(event_windows))
event_windows.head()

event_windows rows after merge: 1080


,event_id,event_date,relative_day,calendar_date,station_name,pm25_ugm3,pm10_ugm3,air_temp_c,rh_pct,wind_speed_ms,wind_dir_deg,grap_stage
0,E001,2022-10-05,-7,2022-09-28,Anand Vihar,90.35,559.60,28.50,70.08,0.42,254.59,0
1,E001,2022-10-05,-7,2022-09-28,Bawana,58.71,125.41,26.89,69.66,0.69,126.64,0
2,E001,2022-10-05,-7,2022-09-28,Jawaharlal Nehru Stadium,45.06,116.61,30.15,65.91,1.28,218.39,0
3,E001,2022-10-05,-7,2022-09-28,Najafgarh,32.89,82.58,30.15,61.32,0.34,140.86,0
4,E001,2022-10-05,-7,2022-09-28,Narela,52.81,141.10,28.37,79.20,1.09,167.76,0


**What this does.** Merges the station-day dataset onto the skeleton, matching
each skeleton row to the station-day row with the same station and calendar
date. `how='left'` keeps every skeleton row regardless of whether a match was
found, so the row count should not change from the skeleton step. The duplicate
date column produced by the merge is then dropped.

**Why event alignment is useful.** This is the step where the "before, during,
after" structure actually gets filled in with real numbers. Because the merge
matches on the exact calendar date, every measurement lands on the correct
relative day automatically, instead of being ordered or joined by hand.

**How analysts use event windows.** This is the point at which an event window
becomes usable: a table where each row is one station's reading on one day
relative to a known policy change, ready to be read window by window in later
notebooks.

In [6]:
# Label each row as before, on, or after the event day, based on relative_day.
event_windows['is_before_event'] = event_windows['relative_day'] < 0
event_windows['is_event_day'] = event_windows['relative_day'] == 0
event_windows['is_after_event'] = event_windows['relative_day'] > 0

event_windows[['event_id', 'relative_day', 'is_before_event', 'is_event_day', 'is_after_event']].head(10)

,event_id,relative_day,is_before_event,is_event_day,is_after_event
0,E001,-7,True,False,False
1,E001,-7,True,False,False
2,E001,-7,True,False,False
3,E001,-7,True,False,False
4,E001,-7,True,False,False
5,E001,-7,True,False,False
6,E001,-7,True,False,False
7,E001,-7,True,False,False
8,E001,-6,True,False,False
9,E001,-6,True,False,False


**What this does.** Adds three True/False columns computed directly from
`relative_day`: whether the row falls before the event, on the event day itself,
or after the event. Exactly one of the three is `True` for any given row.

**Why event alignment is useful.** These flags turn the numeric `relative_day`
into an explicit, readable label. A later notebook that wants "all before-event
rows" can filter on `is_before_event` directly instead of re-deriving it from the
relative day number each time.

**How analysts use event windows.** Before/during/after flags are the most basic
building block of event-window analysis -- later comparisons are built by
grouping or filtering on exactly these three labels, which is why they are
attached once here rather than left for every downstream notebook to recompute.

In [7]:
# Put the columns in a clear, fixed order matching the notebook's specification.
column_order = [
    'event_id',
    'event_date',
    'relative_day',
    'calendar_date',
    'station_name',
    'pm25_ugm3',
    'pm10_ugm3',
    'air_temp_c',
    'rh_pct',
    'wind_speed_ms',
    'wind_dir_deg',
    'grap_stage',
    'is_before_event',
    'is_event_day',
    'is_after_event',
]
event_windows = event_windows[column_order]

# Sort so the table reads naturally: event by event, station by station, day by day.
event_windows = event_windows.sort_values(
    by=['event_id', 'station_name', 'relative_day']
).reset_index(drop=True)

event_windows.head()

,event_id,event_date,relative_day,calendar_date,station_name,pm25_ugm3,pm10_ugm3,air_temp_c,rh_pct,wind_speed_ms,wind_dir_deg,grap_stage,is_before_event,is_event_day,is_after_event
0,E001,2022-10-05,-7,2022-09-28,Anand Vihar,90.35,559.60,28.50,70.08,0.42,254.59,0,True,False,False
1,E001,2022-10-05,-6,2022-09-29,Anand Vihar,97.47,574.12,28.51,67.69,0.46,246.48,0,True,False,False
2,E001,2022-10-05,-5,2022-09-30,Anand Vihar,107.01,554.88,28.62,68.46,0.45,240.65,0,True,False,False
3,E001,2022-10-05,-4,2022-10-01,Anand Vihar,108.54,567.01,28.94,66.04,0.52,210.88,0,True,False,False
4,E001,2022-10-05,-3,2022-10-02,Anand Vihar,94.53,543.08,29.11,62.74,0.69,230.77,0,True,False,False


**What this does.** Reorders the columns into the fixed layout this notebook is
building towards, then sorts the rows so that each event's rows are grouped
together, each station within an event is grouped together, and each station's
15 days run in relative-day order.

**Why event alignment is useful.** A consistent column order and row order is
what makes the master table easy to scan and easy for later notebooks to rely on
without re-sorting or re-selecting columns themselves.

**How analysts use event windows.** Analysts expect an event-window table to
read top to bottom as event, then station, then day -- this ordering is what
makes that expectation hold.

# Section 4 -- Validation

The master table is now built. Before saving it, we check that its structure is
exactly what was intended: the right number of rows, no duplicates, no relative
days outside the +/-7 range, and a complete 15-day window for every station at
every event. Every check below only counts and reports -- nothing is changed or
repaired here.

In [8]:
# Expected rows = number of verified events x number of stations x 15 relative days.
number_of_events = verified_events['event_id'].nunique()
number_of_stations = len(station_list)
number_of_relative_days = len(relative_day_list)

expected_rows = number_of_events * number_of_stations * number_of_relative_days
observed_rows = len(event_windows)

print('Number of verified events:', number_of_events)
print('Number of stations:', number_of_stations)
print('Number of relative days per window:', number_of_relative_days)
print('Expected rows:', expected_rows)
print('Observed rows:', observed_rows)
print('Expected equals observed:', expected_rows == observed_rows)

Number of verified events: 9
Number of stations: 8
Number of relative days per window: 15
Expected rows: 1080
Observed rows: 1080
Expected equals observed: True


**What this does.** Multiplies the number of verified events, the number of
stations, and the number of relative days per window (15) to compute how many
rows the master table should have, then compares that number to the actual row
count of `event_windows`.

**Why event alignment is useful.** Because the table was built from a fixed grid
in Section 2, its row count is fully predictable in advance. Checking the
prediction against the actual count is a simple way to catch a construction
mistake -- a merge that dropped or duplicated rows would show up immediately as
a mismatch here.

**How analysts use event windows.** Analysts trust an event-window table only
once its size has been checked against the expected size. This is normally the
very first thing confirmed before the table is used for anything else.

In [9]:
# Check that every event has exactly 15 distinct relative days.
relative_days_per_event = event_windows.groupby('event_id')['relative_day'].nunique()
relative_days_per_event = relative_days_per_event.reset_index(name='Distinct Relative Days')

# Flag any event that does not have exactly 15.
relative_days_per_event['Has 15 Days'] = relative_days_per_event['Distinct Relative Days'] == 15

relative_days_per_event

,event_id,Distinct Relative Days,Has 15 Days
0,E001,15,True
1,E002,15,True
2,E003,15,True
3,E004,15,True
4,E005,15,True
5,E006,15,True
6,E007,15,True
7,E008,15,True
8,E009,15,True


**What this does.** Groups the master table by `event_id` and counts how many
distinct relative-day values each event has, then flags whether that count
equals the expected 15.

**Why event alignment is useful.** Every event window is supposed to run from -7
to +7 without a gap. Counting the distinct relative days per event directly
checks that promise, one event at a time.

**How analysts use event windows.** An event with fewer than 15 relative days
would mean a shortened window for that event only, which analysts need to know
about before treating every event as equally complete.

In [10]:
# Check that every (event, relative_day) combination has exactly 8 stations.
stations_per_event_day = event_windows.groupby(['event_id', 'relative_day'])['station_name'].nunique()
stations_per_event_day = stations_per_event_day.reset_index(name='Distinct Stations')

# Flag any (event, relative_day) that does not have all 8 stations.
stations_per_event_day['Has All Stations'] = stations_per_event_day['Distinct Stations'] == number_of_stations

# Only show the rows that failed the check, if any.
incomplete_station_days = stations_per_event_day[~stations_per_event_day['Has All Stations']]

print('(event, relative_day) combinations checked:', len(stations_per_event_day))
print('(event, relative_day) combinations missing a station:', len(incomplete_station_days))
incomplete_station_days

(event, relative_day) combinations checked: 135
(event, relative_day) combinations missing a station: 0


,event_id,relative_day,Distinct Stations,Has All Stations


**What this does.** Groups the master table by every (event, relative day) pair
and counts how many distinct stations appear in that group, then keeps only the
groups where the count is below the expected 8.

**Why event alignment is useful.** Every station should contribute exactly one
row to every relative day of every event. This check confirms that no station is
silently missing from part of a window, which would make a later per-station
comparison uneven without an obvious warning sign.

**How analysts use event windows.** A missing station on a given day is a gap an
analyst needs to see before comparing stations against each other -- this table
is where that gap would surface.

In [11]:
# Check for duplicate rows: the same event, station, and relative day should
# never appear more than once.
duplicate_mask = event_windows.duplicated(subset=['event_id', 'station_name', 'relative_day'], keep=False)
duplicate_rows = event_windows[duplicate_mask]

print('Duplicate (event_id, station_name, relative_day) rows:', len(duplicate_rows))
duplicate_rows

Duplicate (event_id, station_name, relative_day) rows: 0


,event_id,event_date,relative_day,calendar_date,station_name,pm25_ugm3,pm10_ugm3,air_temp_c,rh_pct,wind_speed_ms,wind_dir_deg,grap_stage,is_before_event,is_event_day,is_after_event


**What this does.** Flags any row whose (event, station, relative day)
combination is shared with another row, and shows those rows if any exist.

**Why event alignment is useful.** The grid built in Section 2 is meant to have
exactly one row per (event, station, relative day) combination. A duplicate
would mean that combination was double-counted, which would distort any later
count or summary built from this table.

**How analysts use event windows.** A clean event-window table has zero
duplicates by construction. Checking for them directly, rather than assuming the
construction worked, is what makes that assumption verifiable rather than
implicit.

In [12]:
# Check that no relative_day value falls outside the intended -7 to +7 range.
outside_range_mask = (event_windows['relative_day'] < -7) | (event_windows['relative_day'] > 7)
rows_outside_range = event_windows[outside_range_mask]

print('Rows with relative_day outside -7 to +7:', len(rows_outside_range))
rows_outside_range

Rows with relative_day outside -7 to +7: 0


,event_id,event_date,relative_day,calendar_date,station_name,pm25_ugm3,pm10_ugm3,air_temp_c,rh_pct,wind_speed_ms,wind_dir_deg,grap_stage,is_before_event,is_event_day,is_after_event


**What this does.** Flags any row whose `relative_day` value falls outside the
intended -7 to +7 range, and shows those rows if any exist.

**Why event alignment is useful.** The +/-7 day window defined in the project's
analysis plan is only meaningful if every row genuinely falls inside it. This
check confirms the relative-day construction in Section 2 did not accidentally
produce a day outside that boundary.

**How analysts use event windows.** Analysts rely on `relative_day` being
trustworthy on its own, without needing to re-check it against the window
definition every time it is used -- this check is what earns that trust.

In [13]:
# Check for missing measurements: rows where the merge in Section 3 found no
# matching station-day data for that calendar date.
missing_measurements = event_windows[event_windows['pm25_ugm3'].isnull()]

# Break this down by event, so any event with an unusually large gap stands out.
missing_by_event = missing_measurements.groupby('event_id').size()
missing_by_event = missing_by_event.reset_index(name='Rows Missing PM2.5')

print('Total rows with missing PM2.5:', len(missing_measurements))
missing_by_event

Total rows with missing PM2.5: 2


,event_id,Rows Missing PM2.5
0,E007,2


**What this does.** Finds every row where PM2.5 is empty after the Section 3
merge -- meaning the skeleton expected a station-day reading on that calendar
date, but the station-day dataset did not have one -- and counts how many such
rows fall under each event.

**Why event alignment is useful.** Because the skeleton in Section 2 was built
independently of what data actually exists, a missing calendar date shows up
here as an explicit `NaN` row rather than a silently shortened window. This is
exactly the behaviour Section 2's markdown described as the reason to build the
skeleton first.

**How analysts use event windows.** Analysts need to know which event windows
are missing measurements, and how many, before treating every window as equally
reliable. This table is the record they would check first.

In [14]:
# Gather every check above into one validation summary table.
validation_summary = pd.DataFrame({
    'Check': [
        'Expected rows',
        'Observed rows',
        'Events with exactly 15 relative days',
        '(event, relative_day) combinations missing a station',
        'Duplicate (event, station, relative_day) rows',
        'Rows with relative_day outside -7 to +7',
        'Rows with missing PM2.5',
    ],
    'Result': [
        expected_rows,
        observed_rows,
        (relative_days_per_event['Has 15 Days']).sum(),
        len(incomplete_station_days),
        len(duplicate_rows),
        len(rows_outside_range),
        len(missing_measurements),
    ],
})

validation_summary

,Check,Result
0,Expected rows,1080
1,Observed rows,1080
2,Events with exactly 15 relative days,9
3,"(event, relative_day) combinations missing a s...",0
4,"Duplicate (event, station, relative_day) rows",0
5,Rows with relative_day outside -7 to +7,0
6,Rows with missing PM2.5,2


**What this does.** Collects the key numbers from every check above -- expected
versus observed row counts, how many events have a complete 15-day window, how
many (event, day) combinations are missing a station, duplicate rows, out-of-
range days, and rows with missing PM2.5 -- into one summary table.

**Why event alignment is useful.** A single summary table makes it possible to
confirm, at a glance, that the master table's structure matches its
specification before it is saved and handed to a later notebook.

**How analysts use event windows.** This is the table an analyst would look at
first when picking up `event_windows_master.csv` in a future notebook, to see
whether the table's structure is complete or whether specific gaps (missing
stations, missing measurements) need to be kept in mind.

# Section 5 -- Save the Master Table

With the structure validated, the master event-window table is saved for use in
later notebooks.

In [15]:
output_path = '../data/processed/event_windows_master.csv'
event_windows.to_csv(output_path, index=False)

print('Saved:', output_path)
print('Rows saved:', len(event_windows))
print('Columns saved:', list(event_windows.columns))

Saved: ../data/processed/event_windows_master.csv
Rows saved: 1080
Columns saved: ['event_id', 'event_date', 'relative_day', 'calendar_date', 'station_name', 'pm25_ugm3', 'pm10_ugm3', 'air_temp_c', 'rh_pct', 'wind_speed_ms', 'wind_dir_deg', 'grap_stage', 'is_before_event', 'is_event_day', 'is_after_event']


**What this does.** Writes the validated master table to
`data/processed/event_windows_master.csv` and prints the file path, row count,
and column names as a final confirmation of what was saved.

**Why event alignment is useful.** Saving the aligned table once, in one place,
means every later notebook that needs an event window can read the same file
instead of rebuilding the skeleton-and-merge process from scratch.

**How analysts use event windows.** This file is now the shared starting point
for any later before/after comparison, event-consistency check, or sensitivity
analysis described in the project's analysis plan -- none of which are performed
in this notebook.

*End of the Event-Window Construction notebook.*